# 04_SES_Feature_Engineering_Final_v3


In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.preprocessing import MinMaxScaler
pd.set_option('display.max_columns',None)


In [2]:
linkedin = pd.read_csv('linkedin_job_postings.csv',low_memory=False)
postings = pd.read_csv('postings.csv',low_memory=False)
job_skills = pd.read_csv('job_skills.csv',low_memory=False)
stack = pd.read_csv('survey_results_public.csv',low_memory=False)


In [3]:
TECH_SKILLS=[
'python','sql','javascript','typescript','java','c++','c#','go','rust',
'react','next.js','node.js','docker','kubernetes',
'aws','azure','gcp','mongodb','postgresql','mysql','redis',
'tensorflow','pytorch','scikit-learn','spark',
'power bi','tableau','pandas','numpy',
'langchain','llm','rag','fastapi','flask','django'
]


In [4]:
postings['text']=(
postings['title'].fillna('')+' '+
postings['description'].fillna('')+' '+
postings['skills_desc'].fillna('')
).str.lower()

postings[['title','normalized_salary']].head()


,title,normalized_salary
0,Marketing Coordinator,38480.0
1,Mental Health Therapist/Counselor,83200.0
2,Assitant Restaurant Manager,55000.0
3,Senior Elder Law / Trusts and Estates Associat...,157500.0
4,Service Technician,70000.0


In [5]:
from collections import Counter
import pandas as pd

demand_counter = Counter()

for skills in job_skills['job_skills'].dropna():

    skill_list = [
        x.strip().lower()
        for x in str(skills).split(',')
    ]

    for skill in TECH_SKILLS:

        if skill.lower() in skill_list:
            demand_counter[skill] += 1

demand_df = pd.DataFrame(
    demand_counter.items(),
    columns=['skill','linkedin_demand']
)

demand_df.sort_values(
    'linkedin_demand',
    ascending=False
).head(20)

,skill,linkedin_demand
6,python,24921
0,sql,21685
2,java,13735
5,aws,12534
8,javascript,9357
10,azure,7184
7,tableau,6906
4,kubernetes,6640
15,c++,6624
3,docker,6191


In [6]:
import re

salary_rows = []

for skill in TECH_SKILLS:

    subset = postings[
        postings['text'].str.contains(
            rf'\b{re.escape(skill)}\b',
            case=False,
            regex=True,
            na=False
        )
    ]

    salary_rows.append([
        skill,
        subset['normalized_salary'].median()
    ])

salary_df = pd.DataFrame(
    salary_rows,
    columns=['skill', 'salary_premium']
)

salary_df.sort_values(
    'salary_premium',
    ascending=False
).head(20)

,skill,salary_premium
31,rag,190000.00
22,pytorch,179100.00
29,langchain,173325.00
5,c++,172975.00
21,tensorflow,172150.00
23,scikit-learn,171600.00
32,fastapi,170000.00
8,rust,170000.00
28,numpy,168750.00
30,llm,162500.00


In [7]:
geo_base = job_skills.merge(
    linkedin[['job_link','search_country']],
    on='job_link',
    how='inner'
)

total_countries = geo_base[
    'search_country'
].nunique()

geo_rows = []

for skill in TECH_SKILLS:

    subset = geo_base[
        geo_base['job_skills']
        .str.contains(
            skill,
            case=False,
            na=False
        )
    ]

    countries = subset[
        'search_country'
    ].nunique()

    geo_rows.append([
        skill,
        countries / total_countries
    ])

geo_df = pd.DataFrame(
    geo_rows,
    columns=[
        'skill',
        'geographic_spread'
    ]
)

geo_df.sort_values(
    'geographic_spread',
    ascending=False
).head(20)

,skill,geographic_spread
0,python,1.0
1,sql,1.0
2,javascript,1.0
3,typescript,1.0
4,java,1.0
5,c++,1.0
6,c#,1.0
7,go,1.0
8,rust,1.0
9,react,1.0


In [8]:
SKILL_ALIASES = {

    'amazon web services (aws)': 'aws',
    'microsoft azure': 'azure',
    'google cloud': 'gcp',

    'javascript': 'javascript',
    'typescript': 'typescript',

    'node.js': 'node.js',

    'postgresql': 'postgresql',
    'mysql': 'mysql',

    'c#': 'c#',
    'c++': 'c++'
}

def extract_counts(series):

    counter = Counter()

    for row in series.dropna():

        for item in str(row).split(';'):

            skill = item.strip().lower()

            skill = SKILL_ALIASES.get(
                skill,
                skill
            )

            counter[skill] += 1

    return counter

In [9]:
usage_cols=[
'LanguageHaveWorkedWith',
'DatabaseHaveWorkedWith',
'PlatformHaveWorkedWith',
'WebframeHaveWorkedWith',
'ToolsTechHaveWorkedWith',
'MiscTechHaveWorkedWith'
]

usage_counter=Counter()

for col in usage_cols:
    usage_counter.update(
        extract_counts(stack[col])
    )

current_usage_df=pd.DataFrame(
usage_counter.items(),
columns=['skill','current_usage']
)

current_usage_df=current_usage_df[
current_usage_df['skill'].isin(TECH_SKILLS)
]

current_usage_df.head()


,skill,current_usage
1,go,8103
3,java,18239
4,javascript,37492
5,python,30719
6,typescript,23150


In [10]:
future_cols=[
'LanguageWantToWorkWith',
'DatabaseWantToWorkWith',
'PlatformWantToWorkWith',
'WebframeWantToWorkWith',
'ToolsTechWantToWorkWith',
'MiscTechWantToWorkWith'
]

future_counter=Counter()

for col in future_cols:
    future_counter.update(
        extract_counts(stack[col])
    )

future_interest_df=pd.DataFrame(
future_counter.items(),
columns=['skill','future_interest']
)

future_interest_df=future_interest_df[
future_interest_df['skill'].isin(TECH_SKILLS)
]

future_interest_df.head()


,skill,future_interest
1,go,13837
3,java,10668
4,javascript,23774
6,python,25047
7,typescript,20239


In [11]:
master=demand_df.merge(
salary_df,on='skill',how='outer'
)

master=master.merge(
geo_df,on='skill',how='outer'
)

master=master.merge(
current_usage_df,on='skill',how='outer'
)

master=master.merge(
future_interest_df,on='skill',how='outer'
)

master.fillna(0,inplace=True)

master.head()


,skill,linkedin_demand,salary_premium,geographic_spread,current_usage,future_interest
0,aws,12534.0,145000.0,1.0,22191.0,18040.0
1,azure,7184.0,137280.0,1.0,12850.0,10304.0
2,c#,4702.0,133600.0,1.0,16318.0,12921.0
3,c++,6624.0,172975.0,1.0,13827.0,10873.0
4,django,379.0,140400.0,1.0,5835.0,4973.0


In [12]:
features=[
'linkedin_demand',
'salary_premium',
'geographic_spread',
'current_usage',
'future_interest'
]

master[features]=MinMaxScaler().fit_transform(
master[features]
)

master.head()


,skill,linkedin_demand,salary_premium,geographic_spread,current_usage,future_interest
0,aws,0.502949,0.571429,1.0,0.591886,0.686924
1,azure,0.288271,0.497905,1.0,0.342740,0.392354
2,c#,0.188676,0.462857,1.0,0.435240,0.492004
3,c++,0.265800,0.837857,1.0,0.368799,0.414020
4,django,0.015208,0.527619,1.0,0.155633,0.189361


In [13]:
master['future_score']=(
0.35*master['future_interest']+
0.25*master['linkedin_demand']+
0.15*master['current_usage']+
0.15*master['salary_premium']+
0.10*master['geographic_spread']
)

master.sort_values(
'linkedin_demand',
ascending=False
).head(30)


,skill,linkedin_demand,salary_premium,geographic_spread,current_usage,future_interest,future_score
23,python,1.000000,0.523810,1.0,0.819348,0.953735,0.885281
31,sql,0.870150,0.379048,1.0,0.818361,0.852943,0.795679
10,java,0.551142,0.539000,1.0,0.486477,0.406214,0.533782
0,aws,0.502949,0.571429,1.0,0.591886,0.686924,0.640658
11,javascript,0.375466,0.428571,1.0,1.000000,0.905262,0.724994
1,azure,0.288271,0.497905,1.0,0.342740,0.392354,0.435488
32,tableau,0.277116,0.310598,1.0,0.000000,0.000000,0.215869
12,kubernetes,0.266442,0.604488,1.0,0.280140,0.502513,0.475184
3,c++,0.265800,0.837857,1.0,0.368799,0.414020,0.492355
5,docker,0.248425,0.571429,1.0,0.779340,1.000000,0.714721


In [14]:
master.to_csv(
'master_skill_dataset_v3.csv',
index=False
)

print('Saved: master_skill_dataset_v3.csv')
print(master.shape)


Saved: master_skill_dataset_v3.csv
(35, 7)
